LLM Setup

In [44]:
from dotenv import load_dotenv
from pathlib import Path
import os
##from openai import OpenAI

root = Path.cwd()
for candidate in [root, root.parent, root.parent.parent]:
    env_path = candidate / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        break

if os.environ.get("OPENAI_API_KEY"):
    print("OPENAI API KEY is set")
else:
    raise ValueError("OPENAI_API_KEY is not set in environment variables")

OPENAI API KEY is set


In [45]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0.9, max_tokens=1000)
llm_openai.invoke("What is the Capital of Italy?").content

'The capital of Italy is Rome (Italian: Roma).'

In [46]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a movie summariser"),
    ("human","Please summarise movie in brief:{input}")]) 

In [47]:
str_parser=StrOutputParser()

##Template needs inputs in the form of Dictionary ,Hence we need to convert Input to Dictionary

In [48]:
from langchain_core.runnables import RunnableLambda,RunnableParallel

def dictionary_maker(text:str)->dict:
    return {"text":text}

dictionary_maker_runnable=RunnableLambda(dictionary_maker)    

### ***Parallel Chain 1

In [49]:
#Task 1

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system","you are a post generator"),
    ("human","Crate a post for following text for linkedIn:{text}")
])

#Task 2

llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0.9, max_tokens=1000)

#Task 3
str_parser = StrOutputParser()

chain_linkedin=linkedin_prompt | llm_openai | str_parser



### ***Parallel Chain 2



In [50]:
def insta_chain(text:dict):

    text = text["text"] 

    #Task 1
    insta_prompt = ChatPromptTemplate.from_messages([
        ("system","you are a post generator"),
        ("human","Crate a post for following text for Instagram:{text}")
    ])
    
    #Task 2
    
    llm_openai = ChatOpenAI(model="gpt-5-mini", temperature=0.9, max_tokens=1000)
    
    #Task 3
     
    str_parser = StrOutputParser()
     
    chain_insta = insta_prompt | llm_openai | str_parser

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)
    

##Final Orchestration

In [51]:

final_chain = (prompt_template | llm_openai | dictionary_maker_runnable | 
               RunnableParallel(branches = {"linkedin":chain_linkedin,"instagram":insta_chain_runnable})
) 

In [52]:
final_chain.invoke("KGF")

{'branches': {'linkedin': 'KGF (K.G.F: Chapter 1) — a stylized, action-packed period drama — tells the story of Rocky, a young man from Mumbai who turns a promise to his dying mother into relentless ambition. Rising through the criminal underworld to the brutal Kolar Gold Fields, he battles oppressive powers and ruthless gangs to seize control — and the film closes on a cliffhanger, promising more.\n\nWhy this resonates beyond cinema:\n- Ambition with purpose can drive extraordinary outcomes.\n- Navigating hostile environments requires grit, strategic risk-taking and decisive leadership.\n- Power and success bring ethical complexity — leadership isn’t just winning, it’s what you stand for afterward.\n- Stories that end on a cliffhanger remind us: every milestone is a step in a longer journey.\n\nFor leaders and teams: which fiction or film best shaped your view of leadership and resilience? Share your picks — I’ll start with KGF. #Leadership #Resilience #Storytelling #KGF #Ambition',
 

##Chain As Runnable

In [53]:
# TAsk 1
def beutify(final_response:dict)->dict:
    linkedin_response = final_response["branches"]["linkedin"]
    instagram_response = final_response["branches"]["instagram"]
    return {"linkedin": linkedin_response, "instagram": instagram_response}

beutify_runnable = RunnableLambda(beutify)   


#Final Chain with beutify

final_chain = final_chain | beutify_runnable

final_chain.invoke("KGF")

{'linkedin': 'Just watched KGF: Chapter 1 (Kannada) — a gritty period action drama that follows Rocky, a poor young man from Bombay who vows to gain power to fulfill his mother’s last wish. As he rises through the underworld and infiltrates the brutal Kolar Gold Fields, Rocky’s story becomes a raw exploration of ambition, power, and the human cost of change.\n\nFor professionals and leaders, the film offers strong takeaways:\n- Purpose fuels perseverance: Rocky’s single-minded drive shows how a clear purpose can propel action even in hostile environments.  \n- Power vs. responsibility: The film forces us to consider how power is used — for exploitation or for liberation.  \n- Leadership under pressure: Bold decisions, calculated risks, and resilience matter when stakes are high.  \n- Systems and social impact: KGF highlights how entrenched systems exploit vulnerable communities — a reminder that sustainable change requires more than individual heroics.\n\nIf you’re looking for a powerf